# Pokémon Legendary Classification

Academic Python reproduction and completion of the RapidMiner workflow `Pokemon prediction.rmp`.

**Research question:** Can Pokémon type and battle statistics distinguish Legendary from non-Legendary Pokémon?

## 1. Libraries and configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

sns.set_theme(style="whitegrid")
RANDOM_STATE = 2001
DATA_PATH = Path(r"E:\data\pokemon.csv")

## 2. Data loading and variable mapping

The original RapidMiner repository used descriptive names. The CSV uses the standard Pokémon column names. The target `Rank` in the RapidMiner process corresponds to `Legendary` in the CSV.

In [ ]:
CATEGORICAL_FEATURES = ["Type 1", "Type 2"]
NUMERIC_FEATURES = ["HP", "Sp. Atk", "Sp. Def", "Speed", "Total"]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = "Legendary"

raw_data = pd.read_csv(DATA_PATH)
required_columns = FEATURES + [TARGET]
missing_columns = sorted(set(required_columns) - set(raw_data.columns))

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

deduplicated_data = raw_data.drop_duplicates().copy()
data = deduplicated_data[required_columns].copy()
data[TARGET] = data[TARGET].astype(bool).astype(int)

print(f"Rows in source CSV: {len(raw_data):,}")
print(f"Duplicate rows removed: {len(raw_data) - len(data):,}")
print(f"Unique observations retained: {len(data):,}")
display(data.head())

## 3. Data-quality assessment

In [ ]:
quality_summary = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing": data.isna().sum(),
    "unique": data.nunique(dropna=True),
})
display(quality_summary)
display(data[NUMERIC_FEATURES].describe().T.round(2))

## 4. Exploratory analysis

Legendary Pokémon form the minority class, so accuracy should not be interpreted alone.

In [ ]:
target_counts = (
    data[TARGET]
    .value_counts()
    .rename(index={0: "Non-Legendary", 1: "Legendary"})
)
display(target_counts.to_frame("Pokémon"))

ax = target_counts.plot(
    kind="bar",
    figsize=(7, 4),
    color=["#4C78A8", "#E45756"],
)
ax.set_title("Legendary Class Distribution")
ax.set_xlabel("")
ax.set_ylabel("Number of Pokémon")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
type_profile = pd.crosstab(
    data["Type 1"],
    data[TARGET],
    normalize="index",
).rename(columns={0: "Non-Legendary", 1: "Legendary"})

type_profile = type_profile.sort_values("Legendary", ascending=False)
display(type_profile.round(3))

plt.figure(figsize=(10, 6))
sns.barplot(x=type_profile["Legendary"], y=type_profile.index, color="#F28E2B")
plt.title("Share of Legendary Pokémon by Primary Type")
plt.xlabel("Legendary share")
plt.ylabel("Primary type")
plt.tight_layout()
plt.show()

## 5. Preprocessing and Decision Tree

Missing secondary types are imputed with the most frequent category and both type variables are one-hot encoded. The Decision Tree parameters reproduce the RapidMiner configuration as closely as scikit-learn allows.

In [ ]:
X = data[FEATURES].copy()
y = data[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessing = ColumnTransformer([
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ("numeric", "passthrough", NUMERIC_FEATURES),
])

tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=13,
    min_samples_leaf=2,
    min_samples_split=3,
    min_impurity_decrease=0.01,
    random_state=RANDOM_STATE,
)

model = Pipeline([
    ("preprocessing", preprocessing),
    ("decision_tree", tree),
])

print(f"Training observations: {len(X_train):,}")
print(f"Holdout observations: {len(X_test):,}")

## 6. Ten-fold cross-validation

Cross-validation is performed on the 80% training partition, matching the connected branch in the RapidMiner XML.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
)

cv_summary = pd.DataFrame({
    "metric": scoring.keys(),
    "mean": [cv_scores[f"test_{metric}"].mean() for metric in scoring],
    "std": [cv_scores[f"test_{metric}"].std(ddof=1) for metric in scoring],
})
display(cv_summary.round(4))

## 7. Final holdout evaluation

The model is fitted on the complete training partition and evaluated once on the untouched 20% holdout set.

In [ ]:
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

holdout_metrics = pd.Series({
    "accuracy": accuracy_score(y_test, predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
    "precision": precision_score(y_test, predictions, zero_division=0),
    "recall": recall_score(y_test, predictions, zero_division=0),
    "f1": f1_score(y_test, predictions, zero_division=0),
    "roc_auc": roc_auc_score(y_test, probabilities),
}, name="holdout_score")

display(holdout_metrics.to_frame().round(4))
print(classification_report(
    y_test,
    predictions,
    target_names=["Non-Legendary", "Legendary"],
    zero_division=0,
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=["Non-Legendary", "Legendary"],
    cmap="Blues",
)
plt.title("Decision Tree — Holdout Confusion Matrix")
plt.tight_layout()
plt.show()

## 8. Feature importance

In [ ]:
feature_names = model.named_steps["preprocessing"].get_feature_names_out()
importance = pd.Series(
    model.named_steps["decision_tree"].feature_importances_,
    index=feature_names,
    name="importance",
).sort_values(ascending=False)

display(importance.head(15).to_frame().round(4))

plt.figure(figsize=(10, 6))
sns.barplot(x=importance.head(15).values, y=importance.head(15).index, color="#59A14F")
plt.title("Top Decision Tree Feature Importances")
plt.xlabel("Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 9. Decision Tree visualisation

Only the first three levels are displayed so that the principal decision rules remain readable.

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    model.named_steps["decision_tree"],
    feature_names=feature_names,
    class_names=["Non-Legendary", "Legendary"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
)
plt.title("Decision Tree — First Three Levels")
plt.tight_layout()
plt.show()

## 10. Conclusions and limitations

- The Python workflow preserves the selected variables, split ratio, cross-validation and Decision Tree settings from RapidMiner.
- The final holdout evaluation completes the unused 20% branch created by the XML workflow.
- Class imbalance makes recall and F1 for Legendary Pokémon important alongside accuracy.
- The model identifies statistical patterns in this dataset; it does not define the concept of a Legendary Pokémon.
- The dataset contains game-character attributes and the analysis is intended for academic demonstration.